In [1]:
#Imports 
import pandas as pd
import numpy as np
from collections import deque, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

#Nuestros imports
from ID3 import ID3Classifier
from NB import NaiveBayes

ruta = "futbol_uruguayo.csv"
ds = pd.read_csv(ruta)

In [ ]:
#Preprocesamiento
#-1 Eligo los atributos del dataset
atributos_p = ["home_ident", "away_ident"]
atributos_p2 = ["home_ident", "away_ident", "historial"]
atributos = ["home_ident","away_ident","nivel_relativo","forma_local_10","forma_visitante_10"]
atributos_p1 = ["home_ident", "away_ident", "historial", "nivel_relativo"]

#0. Genero un split
tscv = TimeSeriesSplit(n_splits=3) #Con kfoldin habria data leakage 

#1. Calculo columna resultado
ds["resultado"] = np.select(
    [ds["gh"] > ds["ga"], ds["gh"] < ds["ga"]], ["G", "P"], default="E"
)
ds["date"] = pd.to_datetime(ds["date"])

#2. Creacion de nuevas columnas de datos: historial, nivelRelativo 
def calcular_atributos(ds):
    # Orden cronológicamente
    ds = ds.sort_values("date").copy()

    # Listas para guardar resultados 
    historial_list = []
    nivel_list = []

    # Diccionario: Idea Matriz de Historial
    historial_enfrentamientos = {}

    for row in ds.itertuples():     
        if row.home_ident < row.away_ident:
            eq1, eq2 = row.home_ident, row.away_ident
            goles1, goles2 = row.gh, row.ga
            local_es_eq1 = True
        else:
            eq1, eq2 = row.away_ident, row.home_ident
            goles1, goles2 = row.ga, row.gh
            local_es_eq1 = False

        clave_cruce = (eq1, eq2)

        #Inicializacion 
        if clave_cruce not in historial_enfrentamientos:
            historial_enfrentamientos[clave_cruce] = deque(maxlen=10)
        historia_previa = historial_enfrentamientos[clave_cruce]

        #Valores predeterminados
        if not historia_previa:
            historial_list.append("Neutro")
            nivel_list.append("Parejo")
        else:
            victorias_local = 0
            derrotas_local = 0
            goles_favor_local = 0
            goles_contra_local = 0

            for g1, g2 in historia_previa:
                if local_es_eq1:
                    gf, gc = g1, g2
                else:
                    gf, gc = g2, g1

                goles_favor_local += gf
                goles_contra_local += gc

                if gf > gc:
                    victorias_local += 1
                elif gf < gc:
                    derrotas_local += 1

            if victorias_local > derrotas_local:
                historial_list.append("Positivo")
            elif derrotas_local > victorias_local:
                historial_list.append("Negativo")
            else:
                historial_list.append("Neutro")

            promedio_gf = goles_favor_local / len(historia_previa)
            promedio_gc = goles_contra_local / len(historia_previa)
            diferencia = promedio_gf - promedio_gc

            if diferencia <= -1.5:
                nivel_list.append("Muy_Inferior")
            elif diferencia <= -0.5:
                nivel_list.append("Inferior")
            elif diferencia < 0.5:
                nivel_list.append("Parejo")
            elif diferencia < 1.5:
                nivel_list.append("Superior")
            else:
                nivel_list.append("Muy_Superior")

        historial_enfrentamientos[clave_cruce].append((goles1, goles2))

    ds["historial"] = historial_list
    ds["nivel_relativo"] = nivel_list

    return ds
ds = calcular_atributos(ds)

# Estado de forma considerando los últimos 10 partidos

def categorizar_forma(puntos):
    if len(puntos) == 0:
        return "Media"

    rendimiento = sum(puntos) / (3 * len(puntos))

    if rendimiento <= 0.20:
        return "Muy_baja"
    elif rendimiento <= 0.40:
        return "Baja"
    elif rendimiento <= 0.60:
        return "Media"
    elif rendimiento <= 0.80:
        return "Alta"
    else:
        return "Muy_alta"


def calcular_estado_forma(ds, ventana=10):
    ds = ds.sort_values("date").copy()

    columna_local = f"forma_local_{ventana}"
    columna_visitante = f"forma_visitante_{ventana}"

    ds[columna_local] = "Media"
    ds[columna_visitante] = "Media"

    historiales = defaultdict(
        lambda: deque(maxlen=ventana)
    )

    for fecha, partidos_fecha in ds.groupby("date", sort=True):

        # Calculamos la forma usando solamente partidos anteriores
        for indice, partido in partidos_fecha.iterrows():
            local = partido["home_ident"]
            visitante = partido["away_ident"]

            ds.at[indice, columna_local] = categorizar_forma(
                historiales[local]
            )

            ds.at[indice, columna_visitante] = categorizar_forma(
                historiales[visitante]
            )

        # Luego incorporamos los resultados de esta fecha
        for _, partido in partidos_fecha.iterrows():
            local = partido["home_ident"]
            visitante = partido["away_ident"]

            if partido["gh"] > partido["ga"]:
                puntos_local = 3
                puntos_visitante = 0
            elif partido["gh"] < partido["ga"]:
                puntos_local = 0
                puntos_visitante = 3
            else:
                puntos_local = 1
                puntos_visitante = 1

            historiales[local].append(puntos_local)
            historiales[visitante].append(puntos_visitante)

    return ds


ds = calcular_estado_forma(ds, ventana=10)

#3. Separacion en datos de Entrenamiento y Test
mask_test = ds["date"].dt.year >= 2024
train = ds[~mask_test]
test = ds[mask_test]

X_train = train[atributos] 
y_train = train["resultado"]

X_test = test[atributos]
y_test = test["resultado"]


#4. Aplico pipeline para el resto del preprocessing
atributes_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer([
    ('atributes', atributes_pipeline, atributos)
])

In [3]:
#Estimador 1 (sklearn - Random Forest)
pipeline_rf = Pipeline([
    ('preprocessing', preprocessing),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    "model__n_estimators": [10, 25, 50, 75, 100, 150],
    "model__max_depth": [1, 3, 5, 7, None],
}

random_rf = RandomizedSearchCV(
    estimator= pipeline_rf,
    param_distributions=param_grid_rf,
    cv=tscv,
    scoring="f1_macro",
    n_jobs=-1,
    n_iter=24,
    random_state=42
)

random_rf.fit(X_train, y_train)

y_pred_rf = random_rf.predict(X_test)

In [4]:
#Estimador 2 (sklearn - Naive Bayes)
pipeline_nb = Pipeline([
    ('preprocessing', preprocessing),
    ('model', CategoricalNB())
])

param_grid_nb = {"model__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]} #Suavizado de Laplace (dependiendo del alpha creo mas o menos instancias de cada partido)

random_nb = RandomizedSearchCV(
    estimator=pipeline_nb, 
    param_distributions=param_grid_nb, 
    cv=tscv, 
    scoring="accuracy",
    n_jobs=-1,
    n_iter=5,
    random_state=42
)
random_nb.fit(X_train, y_train)

y_pred_nb = random_nb.predict(X_test)

In [5]:
#Estimador 3 (nuestro - id3)
param_grid_id3 = {
    "min_info_gain": [0.001, 0.005, 0.01, 0.02, 0.05, 0.1],
    "criterio": ["Ganancia", "GainRatio", "ImpurityReduction"],  
}

random_id3 = RandomizedSearchCV(
    estimator=ID3Classifier(),
    param_distributions=param_grid_id3,
    n_iter=10,
    scoring="f1_macro",
    n_jobs=-1,
    cv=tscv,
    random_state=42,
)

random_id3.fit(X_train, y_train)

y_pred_id3 = random_id3.predict(X_test)

In [6]:
#Estimador 4 (nuestro - Naive Bayes)
param_grid_nnb = {
    "m":[0.5, 1.0, 2.0, 5.0, 10.0],
}

random_nnb = RandomizedSearchCV(
    estimator=NaiveBayes(),
    param_distributions=param_grid_nnb,
    n_iter=5,
    scoring="f1_macro",
    n_jobs=-1,
    cv=tscv,
    random_state=42,
)

random_nnb.fit(X_train, y_train)

y_pred_nnb = random_nnb.predict(X_test)


In [7]:
#Estimador 5 (Resultado mas probable (10 anos))
subset = (ds["date"].dt.year >= 2014) & (ds["date"].dt.year < 2024)
subset_l = ds.loc[subset, "resultado"]

mas_sale = subset.mode()[0]

In [8]:
# Validacion 

def mostrar_validacion(nombre, busqueda):
    mejor_indice = busqueda.best_index_

    print(nombre)
    print("Mejores hiperparámetros:", busqueda.best_params_)
    print("Macro-F1 promedio:", busqueda.best_score_)
    print(
        "Desviación del Macro-F1:",
        busqueda.cv_results_["std_test_score"][mejor_indice]
    )
    print()


mostrar_validacion("Random Forest", random_rf)
mostrar_validacion("Naive Bayes", random_nb)
mostrar_validacion("ID3 propio", random_id3)
mostrar_validacion("Naive Bayes propio", random_nnb)

Random Forest
Mejores hiperparámetros: {'model__n_estimators': 10, 'model__max_depth': None}
Macro-F1 promedio: 0.3865262906063441
Desviación del Macro-F1: 0.010605664896607907

Naive Bayes
Mejores hiperparámetros: {'model__alpha': 0.5}
Macro-F1 promedio: 0.4547017829667843
Desviación del Macro-F1: 0.007725411288939556

ID3 propio
Mejores hiperparámetros: {'min_info_gain': 0.001, 'criterio': 'Ganancia'}
Macro-F1 promedio: 0.38162112469381837
Desviación del Macro-F1: 0.01409366725075521

Naive Bayes propio
Mejores hiperparámetros: {'m': 10.0}
Macro-F1 promedio: 0.3969771510672
Desviación del Macro-F1: 0.007253855974820477



In [8]:
#Estadisticas

#Random forest
print("Mejores hiperparámetros (Random Forest):", random_rf.best_params_)
print("Accuracy (Random Forest):", accuracy_score(y_test, y_pred_rf))
print("\nReporte de clasificación: (Random Forest)\n", classification_report(y_test, y_pred_rf))

#Naive Bayes
print("Mejores hiperparámetros (Naive Bayes):", random_nb.best_params_)
print("Accuracy (Naive Bayes):", accuracy_score(y_test, y_pred_nb))
print("\nReporte de clasificación: (Naive Bayes)\n", classification_report(y_test, y_pred_nb))

#Nuestro Id3
print("Mejores hiperparámetros (ID3):", random_id3.best_params_)
print("Accuracy (ID3):", accuracy_score(y_test, y_pred_id3))
print("\nReporte de clasificación: (ID3)\n", classification_report(y_test, y_pred_id3))

#Nuestro Naive Bayes
print("Mejores hiperparámetros (Nuestro Naive Bayes):", random_nnb.best_params_)
print("Accuracy (Nuestro Naive Bayes):", accuracy_score(y_test, y_pred_nnb))
print("\nReporte de clasificación: (Nuestro Naive Bayes)\n", classification_report(y_test, y_pred_nnb))

#Estimador base
total = (y_test == "G").sum()/len(y_test)
print("Estimador base:", total)


Mejores hiperparámetros (Random Forest): {'model__n_estimators': 100, 'model__max_depth': None}
Accuracy (Random Forest): 0.4608879492600423

Reporte de clasificación: (Random Forest)
               precision    recall  f1-score   support

           E       0.32      0.27      0.29       132
           G       0.50      0.65      0.57       190
           P       0.50      0.39      0.44       151

    accuracy                           0.46       473
   macro avg       0.44      0.44      0.43       473
weighted avg       0.45      0.46      0.45       473

Mejores hiperparámetros (Naive Bayes): {'model__alpha': 1.0}
Accuracy (Naive Bayes): 0.48414376321353064

Reporte de clasificación: (Naive Bayes)
               precision    recall  f1-score   support

           E       0.32      0.14      0.20       132
           G       0.50      0.74      0.60       190
           P       0.52      0.46      0.49       151

    accuracy                           0.48       473
   macro avg   